# SMS Spam Detection

Detect spam messages using **Multinomial Naive Bayes** and text vectorization.

## Problem Statement

Classify SMS messages as spam or ham using text features extracted from the message content.

## Dataset Information

The dataset contains raw SMS messages and labels indicating whether each message is spam or not.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

sns.set_theme(style="whitegrid")

In [ ]:
spam_df = pd.read_csv("spam.csv", encoding="ISO-8859-1")
spam_df = spam_df.rename(columns={"v1": "Target", "v2": "Message"})
spam_df = spam_df.drop(columns=["", "", ""], errors="ignore")
spam_df.head()

In [ ]:
print("Dataset shape:", spam_df.shape)
print("\nMissing values:\n", spam_df.isnull().sum())
print("\nDuplicate rows:", spam_df.duplicated().sum())
spam_df.info()
spam_df.describe()

In [ ]:
spam_df = spam_df.drop_duplicates().copy()
spam_df["Target"] = spam_df["Target"].map({"ham": 0, "spam": 1})
spam_df["Message_Length"] = spam_df["Message"].astype(str).str.len()
spam_df.head()

In [ ]:
plt.figure(figsize=(8, 4))
sns.countplot(data=spam_df, x="Target")
plt.title("Spam vs Ham Distribution")
plt.xlabel("Label")
plt.ylabel("Count")
plt.xticks([0, 1], ["Ham", "Spam"])
plt.show()

plt.figure(figsize=(10, 4))
sns.histplot(data=spam_df, x="Message_Length", hue="Target", bins=40, kde=False)
plt.title("Message Length by Label")
plt.xlabel("Message Length")
plt.ylabel("Count")
plt.show()

In [ ]:
X = spam_df["Message"]
y = spam_df["Target"]
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

## Model Building and Hyperparameter Tuning

Use a text-processing pipeline and tune the Naive Bayes smoothing parameter.

In [ ]:
pipeline = Pipeline([
    ("vectorizer", CountVectorizer()),
    ("classifier", MultinomialNB())
])

param_grid = {
    "vectorizer__ngram_range": [(1, 1), (1, 2)],
    "classifier__alpha": [0.1, 0.5, 1.0]
}

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid_search.fit(X_train, y_train)
print("Best Parameters:", grid_search.best_params_)
print("Best CV Accuracy: {:.2f}%".format(grid_search.best_score_ * 100))

## Model Evaluation

Evaluate the selected model on the test set.

In [ ]:
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

print("Accuracy: {:.2f}%".format(accuracy_score(y_test, y_pred) * 100))
print("Precision: {:.2f}%".format(precision_score(y_test, y_pred) * 100))
print("Recall: {:.2f}%".format(recall_score(y_test, y_pred) * 100))
print("F1 Score: {:.2f}%".format(f1_score(y_test, y_pred) * 100))

print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=["Ham", "Spam"]))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Ham", "Spam"],
    yticklabels=["Ham", "Spam"]
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Spam Detection Confusion Matrix")
plt.show()

## Results

The tuned SMS spam detection pipeline provides a robust baseline for text classification with a well-structured evaluation.